<a href="https://colab.research.google.com/github/oooinr4018-web/-1/blob/main/ESAA_0511_%EC%88%98%EC%83%81%EC%9E%91%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 주제

Santa 2024 대회의 평가 지표인 Perplexity Metric 이해

# 데이터

- solution DataFrame

id: 문장별 고유 ID

text: 원본 문장

- submission DataFrame

id: 문장별 고유 ID

textL 참가자가 단어 순서를 재배열해서 제출한 문장

- 평가에 사용되는 데이터

원본 문장과 제출 문장의 단어 구성을 비교함. 단어의 종류와 개수가 같아야 하며, 단어를 추가하거나 삭제하면 안됨. 최종적으로 제출 문장의 자연스러움을 Perplexity로 평가함.

# 코드 흐름

1. 제출 형식 검증

원본 문장과 제출 문장을 단어 단우로 분리한 뒤, counter()을 사용해 단어의 종류, 개수가 같은지 확인함. 이를 통해 참가자가 단어를 추가하거나 삭제하지 않고, 주어진 단어들의 순서만 바꾸었는지 검증함.

2. 제출 문장 전처리

' '.join(s.split())을 사용해 제출 문장의 불필요한 공백을 정리함. 같은 문장이라도 공백 차이로 인해 평가가 달라지지 않도록 입력 형식을 통일함.

3. 언어모델 및 토크나이저 로드

perplexitycalculator 클래스에서 Gemma모델과 tokenizer를 불러옴. tokenizer는 문장을 모델이 이해할 수 있는 토큰 단위로 바꾸고, 언어모델은 각 토큰이 등장할 확률을 계산함.

4. Perplexity 계산

각 문장에 시작 토큰과 끝 토큰을 붙인 뒤 토큰화함. 이후 모델의 출력값인 logits와 실제 다음 토큰을 비교하여 Cross Entrpy Loss를 계산함. 이 loss에 exponential을 적용해 Perplexity로 변환함.

5. 최종 점수 산출

제출된 모든 문장의 Perplexity를 계산한 뒤, npmean()을 사용해 평균값을 구함. 이 평균 Perplexity가 최종 평가 점수이며, 값이 낮을수록 더 자연스러운 문장으로 평가됨.








# 주요 코드

In [ ]:
# 사전학습된 언어모델과 tokenizer를 불러오는 부분

self.tokenizer=transformers.AutoTokenizer.from_pretrained(model_path)
self.model=transformers.AutoModelForCausalLM.from(pretrained(...)

# 문장 앞뒤에 시작/끝 토큰을 붙임.
text_with_special=f"{self.tokenizer.bos_token}{text}{self.tokenizer.eos_token}"

# 언어모델은 '다음 단어 예측' 방식이라서, 예측값과 정답 토큰을 한 칸씩 밀어서 비교
shift_logits=logits[...,:-1,:].contiguous()
shift_labels=model_inputs['input_ids'][...,1:].contiguous()

# 모델이 예측한 확률과 실제 다음 토큰 사이의 cross entropy loss를 계산함.
loss=self.loss_fct(
    shift_logits.view(-1,shift_logits.size(-1)),
    shift_labels.view(-1)
)

# loss를 perplexity로 바꾸는 부분.
ppl=[exp(i) for i in loss_list]



# 새롭게 알게 된 내용 / 어려운 내용 / 배울 점

- perplexity는 문장의 자연스러움을 평가하는 지표이다.

- 단어가 같아도 순서가 어색하면 perplexity가 커진다.

- 이 대회는 단어를 새로 만드는 문제가 아니라, 주어진 단어의 순서를 최적화하는 문제이다.

- 언어모델은 다음 토큰을 예측하고, 그 예측이 실제 문장과 얼마나 잘 맞는지를 loss로 계산한다.

- 최종 점수는 각 문장의 perplexity 평균이며, 낮을수록 좋은 점수이다.

- perplexity는 단순 정확도가 아니라 확률 기반 지표이다.

- Tokenizer, Crpss Entropy Loss, Language Model이 함께 사용되는 흐름을 이해하는 것이 어려웠다.

